# 02 · Leave-one-gene-out imputation

For every shared gene: remove it from the spatial data, train the method on the remaining genes, predict the held-out gene.

- **Tangram**: cluster mode, reference grouped by `subclass_name`, uniform density prior
- **SpaGE** (via TISSUE): 20 principal vectors

This is slow (hours for Tangram on all genes). The same step runs headless with
```bash
python scripts/run_loo_imputation.py --dataset d1 --method tangram
```
Output: `data/predictions/tissue_<method>_preds_<dataset>.csv` (cells x genes).

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as st

from spatial_imputation import config, data, metrics, stats, plotting
plotting.set_style()

In [ ]:
from spatial_imputation.imputation import loo_impute

DATASET = "d1"        # "d1" or "d2"
METHOD = "spage"       # "spage" or "tangram"
GENES = None           # e.g. ["nos1", "ngb"] for a quick test; None = all shared genes

In [ ]:
spatial, reference = data.load_processed(DATASET)
spatial

In [ ]:
preds = loo_impute(spatial, reference, METHOD, genes=GENES)
preds.iloc[:5, :5]

In [ ]:
out = config.predictions_path(DATASET, METHOD)
if GENES is None:
    out.parent.mkdir(parents=True, exist_ok=True)
    preds.to_csv(out)
    print('saved', out)

## Quick look

In [ ]:
gene = preds.columns[0]
spatial.obsm[f"{METHOD}_predictions"] = preds.reindex(columns=spatial.var_names)
plotting.plot_actual_vs_predicted(spatial, gene, pred_keys=[f"{METHOD}_predictions"]);